# 03 — Backfill Historical Data

**Mục tiêu:** Cào dữ liệu quá khứ (50 ngày) từ Open-Meteo Archive API → Insert vào TimescaleDB.

**Tại sao cần backfill?**
- LSTM cần ≥ 48 giờ history để forecast
- Isolation Forest cần data quá khứ để học pattern bình thường
- Prophet cần ≥ 30 ngày để detect seasonality

**Số requests ước tính:**
- 63 provinces ÷ 1 request/province = **63 requests** (Archive API, mỗi request = 50 ngày)
- **RẤT ÍT** — nhưng Archive API có thể bị giới hạn trên free tier

**Quy trình:**
1. Test Archive API với 1 tỉnh (HCM)
2. Check data coverage hiện có
3. Backfill 50 ngày cho 63 tỉnh
4. Insert vào TimescaleDB
5. Verify data coverage

## Cell 1: Cài đặt

In [8]:
# Install dependencies nếu cần
# !pip install asyncpg aiohttp pandas -q

import requests
import pandas as pd
import asyncpg
import asyncio
import aiohttp
import json
import time
import pprint
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Cấu hình ──────────────────────────────────────────────────────────
ARCHIVE_URL  = 'https://archive-api.open-meteo.com/v1/archive'
WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'
AQ_URL      = 'https://air-quality-api.open-meteo.com/v1/air-quality'

WEATHER_VARS = 'temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation'
AQ_VARS      = 'pm10,pm2_5,nitrogen_dioxide,ozone,uv_index,us_aqi'

# TimescaleDB
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_USER     = 'reis'
DB_PASSWORD = 'reis_secret'
DB_NAME     = 'reis_db'

# Backfill config
BACKFILL_DAYS = 50  # Số ngày cần backfill

## Cell 2: Check data coverage hiện có

In [9]:
async def check_current_coverage():
    """Kiểm tra data coverage hiện có trong TimescaleDB."""
    conn = await asyncpg.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
    )
    
    rows = await conn.fetch("""
        SELECT 
            province_id,
            COUNT(*)                   AS readings,
            MIN(time)                 AS first_reading,
            MAX(time)                 AS last_reading,
            COUNT(DISTINCT DATE(time)) AS days_covered
        FROM env_readings
        GROUP BY province_id
        ORDER BY province_id
    """)
    await conn.close()
    
    df = pd.DataFrame([dict(r) for r in rows])
    return df

# Chạy check
try:
    df_coverage = await check_current_coverage()
    print(f"Tổng số provinces có data: {len(df_coverage)}")
    display(df_coverage)
    
    if len(df_coverage) > 0:
        print(f"\nTổng readings: {df_coverage['readings'].sum()}")
        print(f"Min days covered: {df_coverage['days_covered'].min()}")
        print(f"Max days covered: {df_coverage['days_covered'].max()}")
        print(f"Mean days covered: {df_coverage['days_covered'].mean():.1f}")
except Exception as e:
    print(f"Lỗi kết nối DB: {e}")
    print("→ TimescaleDB có thể chưa chạy. Chạy: docker-compose up -d")

Tổng số provinces có data: 63


,province_id,readings,first_reading,last_reading,days_covered
0,1,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
1,2,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
2,3,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
3,4,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
4,5,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
...,...,...,...,...,...
58,59,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
59,60,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
60,61,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52
61,62,1224,2026-03-21 17:00:00+00:00,2026-05-11 16:00:00+00:00,52



Tổng readings: 77112
Min days covered: 52
Max days covered: 52
Mean days covered: 52.0


## Cell 3: Test Archive API với HCM (1 tỉnh)

In [10]:
# Test Archive API cho HCM (province_id=2)
# Endpoint: https://archive-api.open-meteo.com/v1/archive

hcm_lat, hcm_lon = 10.7769, 106.7009

# Tính date range: 50 ngày trước
end_date   = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=BACKFILL_DAYS)).strftime('%Y-%m-%d')

print(f"Testing Archive API for HCM")
print(f"Date range: {start_date} → {end_date}")
print(f"Coords: ({hcm_lat}, {hcm_lon})")

# Test Weather Archive
params_weather = {
    'latitude':  hcm_lat,
    'longitude': hcm_lon,
    'start_date': start_date,
    'end_date':   end_date,
    'hourly':     WEATHER_VARS,
    'timezone':   'Asia/Ho_Chi_Minh',
}

r_weather = requests.get(ARCHIVE_URL, params=params_weather, timeout=60)
print(f"\nWeather Archive Status: {r_weather.status_code}")

if r_weather.status_code == 200:
    data_w = r_weather.json()
    print(f"Keys: {list(data_w.keys())}")
    hourly_w = data_w.get('hourly', {})
    print(f"Số giờ data (weather): {len(hourly_w.get('time', []))}")
    print(f"Time range: {hourly_w['time'][0]} → {hourly_w['time'][-1]}")
else:
    print(f"ERROR: {r_weather.text[:200]}")

Testing Archive API for HCM
Date range: 2026-03-23 → 2026-05-12
Coords: (10.7769, 106.7009)

Weather Archive Status: 200
Keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Số giờ data (weather): 1224
Time range: 2026-03-23T00:00 → 2026-05-12T23:00


In [11]:
# Test Air Quality Archive
params_aq = {
    'latitude':  hcm_lat,
    'longitude': hcm_lon,
    'start_date': start_date,
    'end_date':   end_date,
    'hourly':     AQ_VARS,
    'timezone':   'Asia/Ho_Chi_Minh',
}

r_aq = requests.get(AQ_URL, params=params_aq, timeout=60)
print(f"AQ Archive Status: {r_aq.status_code}")

if r_aq.status_code == 200:
    data_aq = r_aq.json()
    hourly_aq = data_aq.get('hourly', {})
    print(f"Số giờ data (AQ): {len(hourly_aq.get('time', []))}")
    
    # Kiểm tra null values
    df_aq_sample = pd.DataFrame(hourly_aq)
    print(f"\nNull counts (HCM AQ sample):")
    display(df_aq_sample.isnull().sum())
else:
    print(f"ERROR: {r_aq.text[:200]}")

AQ Archive Status: 200
Số giờ data (AQ): 1224

Null counts (HCM AQ sample):


time                0
pm10                0
pm2_5               0
nitrogen_dioxide    0
ozone               0
uv_index            0
us_aqi              0
dtype: int64

## Cell 4: Dữ liệu 63 tỉnh

In [12]:
# Dữ liệu 63 tỉnh (từ config/constants.py)
PROVINCES = [
    (1,  'Hà Nội',         21.0285, 105.8542),
    (2,  'Hồ Chí Minh',    10.7769, 106.7009),
    (3,  'Hải Phòng',      20.8623, 106.6799),
    (4,  'Đà Nẵng',        16.0544, 108.2022),
    (5,  'Hà Giang',       22.8279, 104.9823),
    (6,  'Cao Bằng',       22.6761, 106.2016),
    (7,  'Lai Châu',       22.3862, 103.4702),
    (8,  'Lào Cai',        22.4962, 103.9680),
    (9,  'Tuyên Quang',    21.8212, 105.1833),
    (10, 'Lạng Sơn',       22.1398, 105.8320),
    (11, 'Bắc Kạn',        22.1398, 105.8320),
    (12, 'Thái Nguyên',    21.5954, 105.8387),
    (13, 'Yên Bái',        21.7049, 104.8791),
    (14, 'Sơn La',         21.3270, 103.9144),
    (15, 'Phú Thọ',        21.3135, 105.3946),
    (16, 'Vĩnh Phúc',      21.3079, 105.5965),
    (17, 'Quảng Ninh',     20.9489, 107.1035),
    (18, 'Bắc Giang',      21.2804, 106.1985),
    (19, 'Bắc Ninh',       21.2816, 106.1989),
    (20, 'Hải Dương',      20.9411, 106.3330),
    (21, 'Hưng Yên',       20.6626, 106.0585),
    (22, 'Hòa Bình',       20.8199, 105.3438),
    (23, 'Hà Nam',          20.5514, 105.9171),
    (24, 'Nam Định',        20.4272, 106.1749),
    (25, 'Thái Bình',       20.4480, 106.3435),
    (26, 'Ninh Bình',       20.2573, 105.9719),
    (27, 'Thanh Hóa',       19.7996, 105.7864),
    (28, 'Nghệ An',         18.6596, 105.6970),
    (29, 'Hà Tĩnh',         18.3393, 105.9029),
    (30, 'Quảng Bình',      19.6868, 105.7875),
    (31, 'Quảng Trị',       16.7468, 107.1877),
    (32, 'Thừa Thiên Huế',  16.4639, 107.5863),
    (33, 'Quảng Nam',       15.5752, 108.4743),
    (34, 'Quảng Ngãi',      14.3512, 108.0027),
    (35, 'Kon Tum',         13.8865, 109.1133),
    (36, 'Gia Lai',         13.7700, 109.2318),
    (37, 'Bình Định',       13.0467, 109.3108),
    (38, 'Phú Yên',         13.0467, 109.3108),
    (39, 'Đắk Lắk',         12.6797, 108.0447),
    (40, 'Đắk Nông',         12.0006, 107.6960),
    (41, 'Lâm Đồng',         11.9402, 108.4376),
    (42, 'Bình Phước',       11.5314, 106.8943),
    (43, 'Tây Ninh',         10.9460, 106.1900),
    (44, 'Bình Dương',       11.2943, 106.6750),
    (45, 'Đồng Nai',         10.9508, 106.8221),
    (46, 'Bình Thuận',       10.9378, 108.0912),
    (47, 'Khánh Hòa',        11.2349, 109.1941),
    (48, 'Ninh Thuận',       11.5770, 108.9865),
    (49, 'Long An',           10.5389, 106.4061),
    (50, 'Đồng Tháp',        10.3585, 106.3643),
    (51, 'An Giang',          10.3904, 105.4344),
    (52, 'Bà Rịa - Vũng Tàu', 10.4963, 107.1688),
    (53, 'Tiền Giang',       10.3606, 106.3658),
    (54, 'Kiên Giang',        9.9356,  106.3416),
    (55, 'Cần Thơ',           10.0362, 105.7873),
    (56, 'Hậu Giang',         9.7832,  105.4670),
    (57, 'Vĩnh Long',         9.9356,  106.3416),
    (58, 'Bến Tre',           10.2315, 106.3599),
    (59, 'Trà Vinh',          9.9356,  106.3416),
    (60, 'Sóc Trăng',         9.6025,  105.9731),
    (61, 'Bạc Liêu',         9.2869,  105.7228),
    (62, 'Cà Mau',            9.1762,  105.1508),
    (63, 'Điện Biên',        21.3924, 103.0160),
]

print(f"Tổng số tỉnh: {len(PROVINCES)}")
print(f"Backfill: {BACKFILL_DAYS} ngày")
print(f"Số requests ước tính: {len(PROVINCES)} × 2 (weather + AQ) = {len(PROVINCES)*2} requests")

Tổng số tỉnh: 63
Backfill: 50 ngày
Số requests ước tính: 63 × 2 (weather + AQ) = 126 requests


## Cell 5: Backfill cho 1 tỉnh (Hà Nội) — Test trước khi chạy tất cả

In [13]:
async def backfill_province(
    province_id: int,
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    session: aiohttp.ClientSession,
) -> pd.DataFrame:
    """
    Backfill dữ liệu hourly cho 1 tỉnh trong date range.
    
    Gọi song song 2 Archive API (weather + AQ) cho 1 tỉnh.
    Merge theo index (cùng timestamp).
    
    Returns: DataFrame với 1 row mỗi giờ.
    """
    # Weather Archive
    weather_task = session.get(
        ARCHIVE_URL,
        params={
            'latitude': lat, 'longitude': lon,
            'start_date': start_date, 'end_date': end_date,
            'hourly': WEATHER_VARS,
            'timezone': 'Asia/Ho_Chi_Minh',
        },
        timeout=aiohttp.ClientTimeout(total=60),
    )
    
    # AQ Archive (dùng forecast endpoint vì AQ archive có thể không có)
    # Thử archive endpoint trước
    aq_task = session.get(
        AQ_URL,  # Hoặc ARCHIVE_URL nếu AQ archive khả dụng
        params={
            'latitude': lat, 'longitude': lon,
            'start_date': start_date, 'end_date': end_date,
            'hourly': AQ_VARS,
            'timezone': 'Asia/Ho_Chi_Minh',
        },
        timeout=aiohttp.ClientTimeout(total=60),
    )
    
    weather_resp, aq_resp = await asyncio.gather(weather_task, aq_task)
    
    weather_data = await weather_resp.json()
    aq_data     = await aq_resp.json()
    
    # Parse weather
    hw = weather_data.get('hourly', {})
    times_w = hw.get('time', [])
    
    # Parse AQ  
    ha = aq_data.get('hourly', {})
    times_aq = ha.get('time', [])
    
    if not times_w:
        logger.warning(f"Province {province_id}: No weather data returned")
        return pd.DataFrame()
    
    # Build DataFrames và merge trên time index
    df_w  = pd.DataFrame({'time': times_w})
    for key, vals in hw.items():
        if key != 'time':
            df_w[key] = vals
    
    df_aq = pd.DataFrame({'time': times_aq})
    for key, vals in ha.items():
        if key != 'time':
            df_aq[key] = vals
    
    # Merge trên time (inner join — chỉ giữ timestamp có cả 2)
    df = pd.merge(df_w, df_aq, on='time', how='inner')
    
    # Rename columns
    df = df.rename(columns={
        'temperature_2m':       'temperature',
        'relative_humidity_2m': 'humidity',
        'wind_speed_10m':       'wind_speed',
        'nitrogen_dioxide':     'no2',
        'us_aqi':              'aqi',
    })
    
    df['province_id'] = province_id
    
    return df

# ── Test với Hà Nội (province_id=1) ─────────────────────────────────────
hanoi = next(p for p in PROVINCES if p[0] == 1)
pid, pname, plat, plon = hanoi

print(f"Test backfill: {pname} (ID={pid})")

async with aiohttp.ClientSession() as session:
    df_test = await backfill_province(
        province_id=pid,
        lat=plat, lon=plon,
        start_date=start_date,
        end_date=end_date,
        session=session,
    )

print(f"Rows returned: {len(df_test)}")
print(f"Columns: {list(df_test.columns)}")
display(df_test.head(3))
display(df_test.tail(3))

# Null check
print(f"\nNull counts:")
display(df_test.isnull().sum())

Test backfill: Hà Nội (ID=1)
Rows returned: 1224
Columns: ['time', 'temperature', 'wind_speed', 'humidity', 'precipitation', 'pm10', 'pm2_5', 'no2', 'ozone', 'uv_index', 'aqi', 'province_id']


,time,temperature,wind_speed,humidity,precipitation,pm10,pm2_5,no2,ozone,uv_index,aqi,province_id
0,2026-03-23T00:00,22.3,12.0,96,0.0,35.3,32.9,30.3,38.0,0.0,93,1
1,2026-03-23T01:00,21.8,17.3,98,0.0,36.3,33.9,30.2,36.0,0.0,92,1
2,2026-03-23T02:00,22.4,16.2,96,0.1,38.1,35.4,29.5,35.0,0.0,91,1


,time,temperature,wind_speed,humidity,precipitation,pm10,pm2_5,no2,ozone,uv_index,aqi,province_id
1221,2026-05-12T21:00,26.5,14.6,92,0.0,52.6,51.5,51.1,23.0,0.0,116,1
1222,2026-05-12T22:00,26.3,15.2,93,0.0,43.3,42.3,47.2,17.0,0.0,117,1
1223,2026-05-12T23:00,26.1,15.7,93,0.0,36.4,34.5,45.3,15.0,0.0,117,1



Null counts:


time             0
temperature      0
wind_speed       0
humidity         0
precipitation    0
pm10             0
pm2_5            0
no2              0
ozone            0
uv_index         0
aqi              0
province_id      0
dtype: int64

In [14]:
#Query dữ liệu Hà Nội (Province_id = 1)
conn = await asyncpg.connect(
    host=DB_HOST, port=DB_PORT,
    user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
)
# Query 1: Lấy 10 records mới nhất của Hà Nội
rows = await conn.fetch("""
    SELECT time,temperature,humidity,pm2_5, aqi, uv_index 
    FROM env_readings
    WHERE province_id = 1
    ORDER BY time DESC
    LIMIT 10
""")

df = pd.DataFrame([dict(r) for r in rows])
display(df)

,time,temperature,humidity,pm2_5,aqi,uv_index
0,2026-05-11 16:00:00+00:00,24.799999,94.0,44.200001,156,0.00
1,2026-05-11 15:00:00+00:00,25.200001,92.0,45.500000,157,0.00
2,2026-05-11 14:00:00+00:00,25.799999,89.0,48.000000,158,0.00
3,2026-05-11 13:00:00+00:00,26.500000,86.0,49.000000,160,0.00
4,2026-05-11 12:00:00+00:00,27.200001,82.0,48.700001,169,0.00
5,2026-05-11 11:00:00+00:00,28.200001,78.0,46.000000,174,0.00
6,2026-05-11 10:00:00+00:00,29.200001,73.0,30.000000,168,0.30
7,2026-05-11 09:00:00+00:00,30.000000,67.0,30.600000,163,1.30
8,2026-05-11 08:00:00+00:00,30.500000,65.0,33.400002,163,2.90
9,2026-05-11 07:00:00+00:00,30.700001,63.0,36.200001,164,5.35


In [15]:
#Query 2: Thống kê AQI Hà Nội 
stats = await conn.fetch("""
    SELECT 
        COUNT(*)                    AS total_readings,
        AVG(aqi)                    AS avg_aqi,
        MIN(aqi)                    AS min_aqi,
        MAX(aqi)                    AS max_aqi,
        COUNT(DISTINCT DATE(time))  AS days
    FROM env_readings
    WHERE province_id = 1
""")
print(f"Hà Nội - Total readings: {stats[0]['total_readings']}")
print(f"Hà Nội - Days covered: {stats[0]['days']}")
print(f"Hà Nội - Avg AQI: {stats[0]['avg_aqi']:.1f}")
await conn.close()

Hà Nội - Total readings: 1224
Hà Nội - Days covered: 52
Hà Nội - Avg AQI: 138.7


## Cell 6: Insert vào TimescaleDB

In [16]:
_BACKFILL_POOL = None


async def get_backfill_pool() -> asyncpg.Pool:
    global _BACKFILL_POOL
    if _BACKFILL_POOL is None:
        _BACKFILL_POOL = await asyncpg.create_pool(
            host=DB_HOST, port=DB_PORT,
            user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
            min_size=2, max_size=10,
        )
    return _BACKFILL_POOL


async def close_backfill_pool() -> None:
    global _BACKFILL_POOL
    if _BACKFILL_POOL is not None:
        await _BACKFILL_POOL.close()
        _BACKFILL_POOL = None


async def insert_hourly_data(df: pd.DataFrame) -> dict:
    """
    Insert hourly data vao TimescaleDB, dong thoi loc cac timestamp da co.

    Returns:
        {
            'attempted': int,
            'inserted': int,
            'skipped_existing': int,
        }
    """
    if df.empty:
        return {'attempted': 0, 'inserted': 0, 'skipped_existing': 0}

    working = df.copy()
    working['time'] = pd.to_datetime(working['time'], utc=True)
    working = working.sort_values('time').drop_duplicates(subset=['province_id', 'time'])

    province_ids = working['province_id'].dropna().unique().tolist()
    if len(province_ids) != 1:
        raise ValueError(f'insert_hourly_data chi ho tro 1 province/lan, nhan duoc {province_ids}')

    province_id = int(province_ids[0])
    min_time = working['time'].min().to_pydatetime()
    max_time = working['time'].max().to_pydatetime()
    attempted = len(working)

    pool = await get_backfill_pool()

    insert_sql = """
        INSERT INTO env_readings (
            time, province_id,
            temperature, humidity, wind_speed, precipitation,
            pm2_5, pm10, aqi, no2, ozone, uv_index,
            anomaly_score, is_anomaly, raw_json, inserted_at
        ) VALUES (
            $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, $12, $13, $14, $15, $16
        );
    """

    async with pool.acquire() as conn:
        existing_rows = await conn.fetch(
            """
            SELECT time
            FROM env_readings
            WHERE province_id = $1
              AND time >= $2
              AND time <= $3
            """,
            province_id,
            min_time,
            max_time,
        )

        existing_times = {row['time'] for row in existing_rows}
        filtered = working[~working['time'].isin(existing_times)].copy()
        skipped_existing = attempted - len(filtered)

        if filtered.empty:
            return {
                'attempted': attempted,
                'inserted': 0,
                'skipped_existing': skipped_existing,
            }

        rows = []
        now = datetime.now()

        for _, row in filtered.iterrows():
            time_val = row['time']
            if isinstance(time_val, pd.Timestamp):
                time_val = time_val.to_pydatetime()
            elif isinstance(time_val, str):
                time_val = datetime.fromisoformat(time_val.replace('Z', '+00:00'))

            rows.append((
                time_val,
                int(row['province_id']),
                float(row['temperature']) if pd.notna(row.get('temperature')) else None,
                float(row['humidity']) if pd.notna(row.get('humidity')) else None,
                float(row['wind_speed']) if pd.notna(row.get('wind_speed')) else None,
                float(row['precipitation']) if pd.notna(row.get('precipitation')) else None,
                float(row['pm2_5']) if pd.notna(row.get('pm2_5')) else None,
                float(row['pm10']) if pd.notna(row.get('pm10')) else None,
                int(row['aqi']) if pd.notna(row.get('aqi')) else None,
                float(row['no2']) if pd.notna(row.get('no2')) else None,
                float(row['ozone']) if pd.notna(row.get('ozone')) else None,
                float(row['uv_index']) if pd.notna(row.get('uv_index')) else None,
                None,
                False,
                None,
                now,
            ))

        await conn.executemany(insert_sql, rows)

    return {
        'attempted': attempted,
        'inserted': len(rows),
        'skipped_existing': skipped_existing,
    }


# -- Test insert voi Ha Noi -------------------------------------------------
if len(df_test) > 0:
    insert_stats = await insert_hourly_data(df_test)
    print(f'Insert stats for Ha Noi: {insert_stats}')

    conn = await asyncpg.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
    )
    count = await conn.fetchval(
        'SELECT COUNT(*) FROM env_readings WHERE province_id = 1'
    )
    await conn.close()
    print(f'Total rows for Ha Noi in DB: {count}')


Insert stats for Ha Noi: {'attempted': 1224, 'inserted': 31, 'skipped_existing': 1193}
Total rows for Ha Noi in DB: 1255


## Cell 7: BACKFILL TẤT CẢ 63 TỈNH

**⚠️ CHẠY CELL NÀY CẨN THẬN**

- Số requests: 63 provinces × 2 APIs = **126 requests**
- Thời gian ước tính: ~10-20 phút (tùy API rate limit)
- Nếu API rate limit → thêm sleep giữa các request

**Lưu ý:** Nếu Archive API bị limit, chỉ cần backfill **30 ngày** thay vì 50.

In [17]:
def _retry_wait_seconds(attempt: int, error_message: str) -> float:
    if '429' in error_message or 'Too Many Requests' in error_message:
        return float(3 * (2 ** attempt))
    return float(2 ** attempt)


async def backfill_all_provinces_optimized(
    provinces: List,
    start_date: str,
    end_date: str,
    sleep_between: float = 1.5,
    max_retries: int = 3,
    request_timeout_sec: int = 90,
) -> dict:
    """
    Backfill tat ca provinces voi session dung chung, retry ro rang,
    thong ke day du, va tranh insert trung.
    """
    results = {
        'success': [],
        'failed': [],
        'total_rows_fetched': 0,
        'total_rows_inserted': 0,
        'total_rows_skipped_existing': 0,
        'per_province': [],
    }
    start_time = time.time()

    connector = aiohttp.TCPConnector(limit=5, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=request_timeout_sec)

    try:
        async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
            for i, (pid, pname, plat, plon) in enumerate(provinces, 1):
                province_started = time.time()
                province_stats = {
                    'province_id': pid,
                    'province_name': pname,
                    'status': 'failed',
                    'rows_fetched': 0,
                    'rows_inserted': 0,
                    'rows_skipped_existing': 0,
                    'attempts': 0,
                    'error': None,
                }

                print(f'[{i:02d}/{len(provinces)}] {pname:<25} ', end='', flush=True)

                for attempt in range(max_retries):
                    province_stats['attempts'] = attempt + 1
                    try:
                        df = await backfill_province(
                            province_id=pid,
                            lat=plat,
                            lon=plon,
                            start_date=start_date,
                            end_date=end_date,
                            session=session,
                        )

                        province_stats['rows_fetched'] = len(df)
                        results['total_rows_fetched'] += len(df)

                        if df.empty:
                            province_stats['status'] = 'empty'
                            province_stats['error'] = 'No data returned'
                            results['failed'].append(pid)
                            print('WARN no data returned')
                            break

                        insert_stats = await insert_hourly_data(df)
                        province_stats['rows_inserted'] = insert_stats['inserted']
                        province_stats['rows_skipped_existing'] = insert_stats['skipped_existing']
                        province_stats['status'] = 'success'

                        results['success'].append(pid)
                        results['total_rows_inserted'] += insert_stats['inserted']
                        results['total_rows_skipped_existing'] += insert_stats['skipped_existing']

                        elapsed = time.time() - province_started
                        print(
                            f"OK fetched={len(df)}, inserted={insert_stats['inserted']}, "
                            f"skipped_existing={insert_stats['skipped_existing']} "
                            f"({elapsed:.1f}s)"
                        )
                        break

                    except Exception as e:
                        error_message = repr(e)
                        province_stats['error'] = error_message

                        if attempt < max_retries - 1:
                            wait = _retry_wait_seconds(attempt, error_message)
                            print(f'Retry {attempt + 1}/{max_retries} sau {wait:.1f}s...', end=' ', flush=True)
                            await asyncio.sleep(wait)
                        else:
                            results['failed'].append(pid)
                            print(f'ERROR: {error_message}')

                results['per_province'].append(province_stats)

                if i < len(provinces):
                    await asyncio.sleep(sleep_between)
    finally:
        await close_backfill_pool()

    results['total_time_sec'] = time.time() - start_time
    return results


# -- Chay backfill ----------------------------------------------------------
EFFECTIVE_DAYS = 30
effective_start = (datetime.now() - timedelta(days=EFFECTIVE_DAYS)).strftime('%Y-%m-%d')
effective_end = datetime.now().strftime('%Y-%m-%d')

print(f'Backfilling {EFFECTIVE_DAYS} days: {effective_start} -> {effective_end}')
print(f'Provinces: {len(PROVINCES)}')
print(f'Sleep between provinces: {1.5}s')
print('=' * 80)

results = await backfill_all_provinces_optimized(
    provinces=PROVINCES,
    start_date=effective_start,
    end_date=effective_end,
    sleep_between=1.5,
    max_retries=3,
    request_timeout_sec=90,
)

print('' + '=' * 80)
print('BACKFILL COMPLETE!')
print(f"Success provinces           : {len(results['success'])}/{len(PROVINCES)}")
print(f"Failed provinces            : {len(results['failed'])}/{len(PROVINCES)}")
print(f"Total rows fetched          : {results['total_rows_fetched']:,}")
print(f"Total rows inserted         : {results['total_rows_inserted']:,}")
print(f"Total rows skipped existing : {results['total_rows_skipped_existing']:,}")
print(f"Time elapsed                : {results['total_time_sec']/60:.1f} minutes")

if results['failed']:
    print(f"Failed province IDs: {results['failed']}")

results_df = pd.DataFrame(results['per_province'])
display(results_df)


Backfilling 30 days: 2026-04-12 -> 2026-05-12
Provinces: 63
Sleep between provinces: 1.5s
[01/63] Hà Nội                    OK fetched=744, inserted=0, skipped_existing=744 (2.6s)
[02/63] Hồ Chí Minh               OK fetched=744, inserted=31, skipped_existing=713 (0.2s)
[03/63] Hải Phòng                 OK fetched=744, inserted=31, skipped_existing=713 (0.3s)
[04/63] Đà Nẵng                   OK fetched=744, inserted=31, skipped_existing=713 (0.3s)
[05/63] Hà Giang                  OK fetched=744, inserted=31, skipped_existing=713 (0.3s)
[06/63] Cao Bằng                  OK fetched=744, inserted=31, skipped_existing=713 (0.2s)
[07/63] Lai Châu                  OK fetched=744, inserted=31, skipped_existing=713 (2.4s)
[08/63] Lào Cai                   OK fetched=744, inserted=31, skipped_existing=713 (1.6s)
[09/63] Tuyên Quang               OK fetched=744, inserted=31, skipped_existing=713 (0.3s)
[10/63] Lạng Sơn                  OK fetched=744, inserted=31, skipped_existing=713 (0.4s)
[

,province_id,province_name,status,rows_fetched,rows_inserted,rows_skipped_existing,attempts,error
0,1,Hà Nội,success,744,0,744,1,None
1,2,Hồ Chí Minh,success,744,31,713,1,None
2,3,Hải Phòng,success,744,31,713,1,None
3,4,Đà Nẵng,success,744,31,713,1,None
4,5,Hà Giang,success,744,31,713,1,None
...,...,...,...,...,...,...,...,...
58,59,Trà Vinh,success,744,31,713,1,None
59,60,Sóc Trăng,success,744,31,713,1,None
60,61,Bạc Liêu,success,744,31,713,1,None
61,62,Cà Mau,success,744,31,713,1,None


## Cell 8: Verify Data Coverage sau Backfill

In [18]:
# Kiểm tra lại data coverage
df_coverage_after = await check_current_coverage()
print(f"Tổng số provinces có data: {len(df_coverage_after)}")
display(df_coverage_after)

if len(df_coverage_after) > 0:
    print(f"\n📊 Coverage Summary:")
    print(f"  - Min days covered: {df_coverage_after['days_covered'].min()}")
    print(f"  - Max days covered: {df_coverage_after['days_covered'].max()}")
    print(f"  - Mean days covered: {df_coverage_after['days_covered'].mean():.1f}")
    print(f"  - Total readings: {df_coverage_after['readings'].sum():,}")
    
    # Provinces với < 30 ngày data
    low_coverage = df_coverage_after[df_coverage_after['days_covered'] < 30]
    if len(low_coverage) > 0:
        print(f"\n⚠️  Provinces với < 30 ngày data ({len(low_coverage)}):")
        display(low_coverage)
    else:
        print(f"\n✅ Tất cả provinces có ≥ 30 ngày data — sẵn sàng train ML!")

Tổng số provinces có data: 63


,province_id,readings,first_reading,last_reading,days_covered
0,1,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
1,2,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
2,3,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
3,4,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
4,5,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
...,...,...,...,...,...
58,59,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
59,60,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
60,61,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53
61,62,1255,2026-03-21 17:00:00+00:00,2026-05-12 23:00:00+00:00,53



📊 Coverage Summary:
  - Min days covered: 53
  - Max days covered: 53
  - Mean days covered: 53.0
  - Total readings: 79,065

✅ Tất cả provinces có ≥ 30 ngày data — sẵn sàng train ML!


## Cell 9: AQI Distribution Check

In [19]:
async def get_aqi_distribution():
    conn = await asyncpg.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
    )
    rows = await conn.fetch("""
        SELECT 
            province_id,
            AVG(aqi) as avg_aqi,
            MIN(aqi) as min_aqi,
            MAX(aqi) as max_aqi,
            STDDEV(aqi) as std_aqi
        FROM env_readings
        WHERE aqi IS NOT NULL
        GROUP BY province_id
        ORDER BY avg_aqi DESC
    """)
    await conn.close()
    return pd.DataFrame([dict(r) for r in rows])

df_aqi = await get_aqi_distribution()
print(f"AQI Statistics cho {len(df_aqi)} provinces:")
display(df_aqi.head(10))

print(f"\nTop 5 ô nhiễm nhất:")
display(df_aqi.head(5))

print(f"\n5 tỉnh sạch nhất:")
display(df_aqi.tail(5))

AQI Statistics cho 63 provinces:


,province_id,avg_aqi,min_aqi,max_aqi,std_aqi
0,16,150.1521912350597610,82,250,32.6391259845475998
1,15,147.2613545816733068,68,268,32.5830588227468274
2,13,142.9314741035856574,61,234,33.2569392547004544
3,21,142.0709163346613546,74,236,33.9624134165510474
4,18,138.5338645418326693,64,261,36.4465297402815589
5,19,138.5338645418326693,64,261,36.4465297402815589
6,1,138.5338645418326693,64,261,36.4465297402815589
7,9,133.8167330677290837,45,231,37.1744652699473444
8,22,132.4613545816733068,66,228,37.5910577155420653
9,26,129.6517928286852590,64,226,34.1089916616418276



Top 5 ô nhiễm nhất:


,province_id,avg_aqi,min_aqi,max_aqi,std_aqi
0,16,150.1521912350597610,82,250,32.6391259845475998
1,15,147.2613545816733068,68,268,32.5830588227468274
2,13,142.9314741035856574,61,234,33.2569392547004544
3,21,142.0709163346613546,74,236,33.9624134165510474
4,18,138.5338645418326693,64,261,36.4465297402815589



5 tỉnh sạch nhất:


,province_id,avg_aqi,min_aqi,max_aqi,std_aqi
58,60,57.2908366533864542,39,81,6.9288273195062266
59,44,56.9760956175298805,36,80,8.1656144328403190
60,59,55.2876494023904382,40,73,6.2889715479219503
61,46,53.7960159362549801,37,73,7.2250108347838648
62,51,51.1035856573705179,36,129,10.1244463043728497
